# Russell 2000 Rebalance — Long/Short Backtest

## Strategy
Russell reconstitution creates predictable price pressure. We exploit it:

| Signal | Direction | Classification |
|---|---|---|
| Stock moving into R2000 from Microcap | **Long** | `FROM_RMICRO` |
| Stock moving into R2000 from R1000 | **Long** | `FROM_R1000` |
| Stock being demoted to Microcap | **Short** | `TO_RMICRO` |

**Holding period**: Enter 60 trading days before rank date (t = −60). Exit 20 trading days after (t = +20).

**Risk model**: OLS on FF5 (Mkt-RF, SMB, HML, RMW, CMA) estimates per-stock factor loadings **B**  
and idiosyncratic variance **Ωε**. Total covariance **ΩR = B ΩF Bᵀ + Ωε** feeds the MVO vol constraint.  
PCA on OLS residuals validates how much variance is truly idiosyncratic.

**Neutrality**: FF5-factor neutral (Bᵀw = 0) + GICS Sector neutral (Σ_{i∈k} wᵢ = 0).

**Three weighting schemes tested**:
- **A — Equal weight**: ±1 per stock, optimizer enforces all constraints  
- **B — Market-cap weight**: signal ∝ MCAP at entry  
- **C — Confidence weight**: signal ∝ predicted-rank distance from R2000 boundary (from Factor-ARIMA notebook)

Signal mode: **hindsight** (known events). Displays theoretical max alpha; flags lookahead bias.

In [ ]:
import warnings, pickle
from pathlib import Path

import numpy as np
import pandas as pd
import statsmodels.api as sm
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy.optimize import minimize
from scipy.stats import spearmanr

sns.set_theme(style='whitegrid', font_scale=1.05)
warnings.filterwarnings('ignore')

# ── Paths ─────────────────────────────────────────────────────────────────────
PROCESSED_DIR  = Path('../../data/processed')
RAW_MBR_DIR    = Path('../../data/raw/membership')
FF5_FILE       = PROCESSED_DIR / 'F-F_Research_Data_5_Factors_2x3_daily.csv'
PRICES_PARQUET = PROCESSED_DIR / 'FINALIZED_PRICES.parquet'
MCAPS_PARQUET  = PROCESSED_DIR / 'FINALIZED_MCAPS.parquet'
CALENDAR_FILE  = RAW_MBR_DIR   / 'russell_calendar.xlsx'
EVENTS_FILE    = PROCESSED_DIR / 'russell_events.csv'
GICS_FILE      = PROCESSED_DIR / 'RUSSELL_INDUSTRIES.csv'
FA_CACHE_DIR   = PROCESSED_DIR / 'factor_arima_cache'

# ── Strategy parameters ───────────────────────────────────────────────────────
FACTOR_COLS   = ['Mkt-RF', 'SMB', 'HML', 'RMW', 'CMA']
LONG_CLASSES  = ['FROM_RMICRO', 'FROM_R1000']
SHORT_CLASSES = ['TO_RMICRO']
ENTRY_OFFSET  = -60    # trading days before rank date
EXIT_OFFSET   =  20    # trading days after rank date
TRAIN_DAYS    = 252    # OLS estimation window
MIN_OBS       =  60    # min observations for OLS
MIN_SECTOR_N  =   3    # min stocks in sector to enforce neutrality
VOL_CAP_ANN   = 0.10   # annualised vol cap
GROSS         = 2.0    # normalised gross exposure ($1 long + $1 short)

SCHEMES = ['A_equal', 'B_mktcap', 'C_confidence']
SCHEME_LABELS = {'A_equal': 'A: Equal weight',
                 'B_mktcap': 'B: Market-cap weight',
                 'C_confidence': 'C: Confidence weight'}
SCHEME_COLORS = {'A_equal': '#1565C0', 'B_mktcap': '#2E7D32', 'C_confidence': '#E65100'}

print('Parameters set.')
print(f'Entry: t={ENTRY_OFFSET}  Exit: t=+{EXIT_OFFSET}  Train: {TRAIN_DAYS}d  Vol cap: {VOL_CAP_ANN:.0%}')

In [ ]:
# ── FF5 factors ────────────────────────────────────────────────────────────────
FF5 = pd.read_csv(
    FF5_FILE, skiprows=2, header=None,
    names=['Date', 'Mkt-RF', 'SMB', 'HML', 'RMW', 'CMA', 'RF']
)
FF5['Date'] = pd.to_datetime(FF5['Date'].astype(str), format='%Y%m%d')
FF5 = FF5.set_index('Date') / 100
print(f'FF5   : {FF5.shape[0]:,} days  ({FF5.index.min().date()} → {FF5.index.max().date()})')

# ── Daily prices → returns, aligned to FF5 calendar ───────────────────────────
prices  = pd.read_parquet(PRICES_PARQUET)
returns = prices.pct_change(fill_method=None)
returns = returns[returns.index.isin(FF5.index)]
print(f'Returns: {returns.shape[0]:,} dates × {returns.shape[1]:,} tickers')

# ── Market caps ────────────────────────────────────────────────────────────────
mcaps = pd.read_parquet(MCAPS_PARQUET)
print(f'MCaps  : {mcaps.shape[0]:,} dates × {mcaps.shape[1]:,} tickers')

# ── Russell calendar ───────────────────────────────────────────────────────────
_cal = pd.read_excel(CALENDAR_FILE, sheet_name='Russell Calendar', header=3)
_cal.columns = ['Year', 'Period', 'Rank_Date', 'Announcement_Date',
                'Effective_Date', 'Effective_Open', 'Notes']
calendar = (
    _cal[['Year', 'Rank_Date', 'Effective_Date']]
    .dropna(subset=['Year'])
    .assign(
        Year           = lambda d: d['Year'].astype(int),
        Rank_Date      = lambda d: pd.to_datetime(d['Rank_Date']),
        Effective_Date = lambda d: pd.to_datetime(d['Effective_Date']),
    )
    .set_index('Year')
)
print(f'Calendar: {len(calendar)} years ({calendar.index.min()} – {calendar.index.max()})')

# ── Reconstitution events ──────────────────────────────────────────────────────
events = pd.read_csv(EVENTS_FILE)
events['Rank_Date'] = pd.to_datetime(events['Rank_Date'])
print(f'Events : {len(events):,} rows | {events["Classification"].value_counts().to_dict()}')

# ── GICS sector map (ticker → sector) ─────────────────────────────────────────
# GICS tickers lack the " Equity" suffix → add it to match prices/events format
gics_raw = pd.read_csv(GICS_FILE)
gics_valid = gics_raw[
    gics_raw['GICS Sector'].notna() & (gics_raw['GICS Sector'] != '#N/A N/A')
].copy()
gics_valid['Ticker_full'] = gics_valid['Ticker'].str.strip() + ' Equity'
sector_map = gics_valid.set_index('Ticker_full')['GICS Sector'].to_dict()
print(f'GICS   : {len(sector_map):,} tickers with valid sector | '
      f'{gics_valid["GICS Sector"].nunique()} unique sectors')

In [ ]:
# ── Confidence scores from Factor-ARIMA cache ──────────────────────────────────
# Confidence = predicted-rank distance from nearest R2000 boundary (1000 or 2001),
# normalised by universe size. Stocks deep inside or outside R2000 → high confidence.
confidence_store = {}   # year → {ticker: float [0,1]}

for year in range(2006, 2026):
    cache_file = FA_CACHE_DIR / f'year_{year}.pkl'
    if not cache_file.exists():
        continue
    try:
        with open(cache_file, 'rb') as f:
            cached = pickle.load(f)
        cmp = cached['cmp'].copy()
        # Rank by predicted mcap (descending = rank 1 is largest)
        cmp = cmp.sort_values('predicted', ascending=False)
        cmp['pred_rank'] = np.arange(1, len(cmp) + 1)
        n = len(cmp)
        # Distance from nearest R2000 boundary (ranks 1001–2000)
        cmp['conf'] = cmp['pred_rank'].apply(
            lambda r: min(abs(r - 1000), abs(r - 2001)) / max(n, 1)
        )
        confidence_store[year] = cmp['conf'].to_dict()
    except Exception as e:
        print(f'[{year}] confidence load failed: {e}')

print(f'Confidence scores loaded for {len(confidence_store)} years.')
if confidence_store:
    sample_yr = min(confidence_store.keys())
    scores = list(confidence_store[sample_yr].values())
    print(f'  Sample year {sample_yr}: n={len(scores):,} | '
          f'mean={np.mean(scores):.3f} | max={np.max(scores):.3f}')

In [ ]:
# ── Helper: trading date offset ────────────────────────────────────────────────
def trading_date_at(trading_dates, ref_date, offset=0):
    """Return the trading date 'offset' days from ref_date.
    offset=-60 → 60 trading days before ref_date.
    """
    idx = trading_dates.searchsorted(ref_date)
    new_idx = int(np.clip(idx + offset, 0, len(trading_dates) - 1))
    return trading_dates[new_idx]


# ── FF5 factor model estimation ────────────────────────────────────────────────
def estimate_ff5_model(tickers, ret_train, ff5_train):
    """
    OLS regression of excess returns on FF5 for each ticker.

    Returns
    -------
    B         : ndarray (n, 5)  — factor loadings (betas only, no intercept)
    omega_eps : ndarray (n,)    — idiosyncratic daily variance per stock
    Omega_F   : ndarray (5, 5)  — FF5 sample covariance
    r2        : ndarray (n,)    — OLS R²
    valid     : bool ndarray (n,) — True where OLS converged with enough data
    """
    n = len(tickers)
    B         = np.zeros((n, 5))
    omega_eps = np.zeros(n)
    r2        = np.zeros(n)
    valid     = np.zeros(n, dtype=bool)

    RF = ff5_train['RF'].values
    X_factors = ff5_train[FACTOR_COLS].values          # (T, 5)
    X_ols     = np.column_stack([np.ones(len(X_factors)), X_factors])  # (T, 6)

    for i, tkr in enumerate(tickers):
        if tkr not in ret_train.columns:
            continue
        y_raw = ret_train[tkr].values
        exc   = y_raw - RF            # excess return
        mask  = np.isfinite(exc)
        if mask.sum() < MIN_OBS:
            continue
        y_exc = exc[mask]
        X_m   = X_ols[mask]
        try:
            coef, _, _, _ = np.linalg.lstsq(X_m, y_exc, rcond=None)
        except Exception:
            continue
        y_hat = X_m @ coef
        resid = y_exc - y_hat
        ss_res = resid @ resid
        ss_tot = ((y_exc - y_exc.mean()) ** 2).sum()
        B[i]         = coef[1:]                          # drop intercept
        omega_eps[i] = ss_res / max(mask.sum() - 6, 1)  # per-day idio variance
        r2[i]        = 1 - ss_res / ss_tot if ss_tot > 0 else 0.0
        valid[i]     = True

    Omega_F = ff5_train[FACTOR_COLS].cov().values  # (5, 5)
    return B, omega_eps, Omega_F, r2, valid


# ── PCA on OLS residuals (diagnostic) ─────────────────────────────────────────
def pca_residual_diagnostics(tickers, ret_train, ff5_train, B, valid):
    """
    Fit residual returns for valid stocks and compute PCA explained variance.
    Returns explained variance ratios (array, up to 10 components).
    """
    RF = ff5_train['RF'].values
    X_factors = ff5_train[FACTOR_COLS].values

    resid_cols = []
    for i, tkr in enumerate(tickers):
        if not valid[i] or tkr not in ret_train.columns:
            continue
        y_raw = ret_train[tkr].values
        exc   = y_raw - RF
        mask  = np.isfinite(exc)
        if mask.sum() < MIN_OBS:
            continue
        pred  = X_factors[mask] @ B[i]
        resid = exc[mask] - pred
        s     = pd.Series(resid, name=tkr)
        resid_cols.append(s)

    if len(resid_cols) < 10:
        return np.array([])

    resid_df = pd.concat(resid_cols, axis=1).dropna(axis=1)
    cov_r    = resid_df.cov().values
    eigvals  = np.linalg.eigvalsh(cov_r)[::-1]   # descending
    eigvals  = eigvals[eigvals > 0]
    return eigvals[:10] / eigvals.sum()


print('Helper functions defined.')

In [ ]:
# ── Portfolio construction via constrained MVO ─────────────────────────────────
def build_weights(mu, B, omega_eps, Omega_F, sectors,
                  gross=GROSS, vol_cap_ann=VOL_CAP_ANN, trading_days=252):
    """
    Solve: maximise wᵀμ subject to:
      • dollar neutral   : Σw = 0
      • factor neutral   : Bᵀw = 0  (5 constraints)
      • sector neutral   : Σ_{i∈k} wᵢ = 0  per GICS sector (≥ MIN_SECTOR_N stocks)
      • vol cap          : wᵀΩR w ≤ vol_cap_ann²/trading_days

    After optimisation, normalise so Σ|w| = gross.

    Parameters
    ----------
    mu        : (n,)   signal vector (positive = long, negative = short)
    B         : (n,5)  FF5 factor loadings
    omega_eps : (n,)   idiosyncratic daily variances
    Omega_F   : (5,5)  FF5 factor covariance
    sectors   : list of str or None, length n

    Returns
    -------
    w : (n,) portfolio weights, normalised to Σ|w|=gross.
        Returns simple normalised mu fallback on optimisation failure.
    """
    n = len(mu)
    daily_var_cap = (vol_cap_ann ** 2) / trading_days

    # Systematic covariance component: B Ω_F Bᵀ
    BF  = B @ Omega_F          # (n, 5)
    BFBt = BF @ B.T            # (n, n) — systematic
    Omega_eps_diag = np.diag(np.where(np.isfinite(omega_eps), omega_eps, 0))
    Omega_R = BFBt + Omega_eps_diag

    # ── Constraints ───────────────────────────────────────────────────────────
    constraints = []

    # Dollar neutral
    constraints.append({'type': 'eq', 'fun': lambda w: np.sum(w),
                        'jac': lambda w: np.ones(n)})

    # FF5 factor neutral: B[:,k]ᵀ w = 0 for each k
    for k in range(5):
        bk = B[:, k].copy()
        constraints.append({'type': 'eq',
                            'fun': lambda w, b=bk: b @ w,
                            'jac': lambda w, b=bk: b})

    # Sector neutral (only sectors with ≥ MIN_SECTOR_N stocks)
    sec_to_idx = {}
    for i, sec in enumerate(sectors):
        if sec is not None and str(sec) not in ('nan', 'None'):
            sec_to_idx.setdefault(sec, []).append(i)
    for sec, idxs in sec_to_idx.items():
        if len(idxs) >= MIN_SECTOR_N:
            mask = np.zeros(n)
            mask[idxs] = 1.0
            constraints.append({'type': 'eq',
                                'fun': lambda w, m=mask: m @ w,
                                'jac': lambda w, m=mask: m})

    # Vol cap (quadratic inequality)
    constraints.append({'type': 'ineq',
                        'fun': lambda w: daily_var_cap - float(w @ Omega_R @ w),
                        'jac': lambda w: -2.0 * (Omega_R @ w)})

    # ── Warm start: normalised signal ─────────────────────────────────────────
    abs_mu = np.abs(mu)
    w0 = mu / abs_mu.sum() * gross if abs_mu.sum() > 0 else mu

    # Bounds: each weight within ±gross (loose; vol cap is the binding constraint)
    bounds = [(-gross, gross)] * n

    result = minimize(
        fun=lambda w: -float(mu @ w),
        x0=w0,
        method='SLSQP',
        bounds=bounds,
        constraints=constraints,
        options={'maxiter': 2000, 'ftol': 1e-9},
    )

    if result.success:
        w = result.x
    else:
        # Fallback: normalised raw signal
        w = w0

    # Normalise gross exposure
    g = np.sum(np.abs(w))
    if g > 1e-8:
        w = w / g * gross

    return w, result.success


print('build_weights() defined.')

In [ ]:
# ── Per-year simulation ────────────────────────────────────────────────────────
def simulate_year(year, row):
    """
    Run the full backtest for one rebalance year.

    Returns
    -------
    pnl_df   : DataFrame indexed by trading date, columns = scheme names → daily P&L
               plus columns factor_<name> (exposure = Bᵀw per factor, at entry).
    meta     : dict of year-level metadata
    exp_drift: DataFrame (date × factor) of realised factor exposures over holding period
               for scheme A (equal weight) — used in drift plot.
    """
    rank_date = row['Rank_Date']
    trd_idx   = returns.index

    entry_date = trading_date_at(trd_idx, rank_date, ENTRY_OFFSET)
    exit_date  = trading_date_at(trd_idx, rank_date, EXIT_OFFSET)

    # ── Event stocks ──────────────────────────────────────────────────────────
    yr_ev     = events[events['Year'] == year]
    long_tkrs = yr_ev[yr_ev['Classification'].isin(LONG_CLASSES)]['Ticker'].tolist()
    short_tkrs= yr_ev[yr_ev['Classification'].isin(SHORT_CLASSES)]['Ticker'].tolist()
    all_tkrs  = list(dict.fromkeys(long_tkrs + short_tkrs))  # dedup, preserve order

    # Keep only tickers present in the prices parquet
    all_tkrs  = [t for t in all_tkrs if t in returns.columns]
    long_set  = set(long_tkrs) & set(all_tkrs)
    short_set = set(short_tkrs) & set(all_tkrs)

    if len(long_set) < 3 or len(short_set) < 3:
        return None, {'year': year, 'skip': 'too few stocks with price data'}, None

    # ── Training window ────────────────────────────────────────────────────────
    ret_train_all = returns[(returns.index <= entry_date)].iloc[-TRAIN_DAYS:]
    ff5_train     = FF5[(FF5.index >= ret_train_all.index.min()) &
                        (FF5.index <= entry_date)]
    ret_train     = ret_train_all[all_tkrs]

    # ── FF5 factor model ───────────────────────────────────────────────────────
    B, omega_eps, Omega_F, r2, valid = estimate_ff5_model(all_tkrs, ret_train, ff5_train)

    # Remove stocks where OLS failed
    all_tkrs  = [t for t, v in zip(all_tkrs, valid) if v]
    B         = B[valid]
    omega_eps = omega_eps[valid]
    r2        = r2[valid]
    long_set  = long_set & set(all_tkrs)
    short_set = short_set & set(all_tkrs)

    if len(long_set) < 3 or len(short_set) < 3:
        return None, {'year': year, 'skip': 'too few stocks with valid OLS'}, None

    n = len(all_tkrs)

    # ── PCA on residuals (diagnostic) ─────────────────────────────────────────
    pca_evr = pca_residual_diagnostics(all_tkrs, ret_train, ff5_train, B, np.ones(n, bool))

    # ── GICS sector labels ────────────────────────────────────────────────────
    sectors = [sector_map.get(t) for t in all_tkrs]

    # ── Market cap at entry ───────────────────────────────────────────────────
    mcap_snap = mcaps[mcaps.index <= entry_date].iloc[-1]

    # ── Confidence scores at entry ────────────────────────────────────────────
    yr_conf = confidence_store.get(year, {})

    # ── Build signal vectors for each scheme ──────────────────────────────────
    def make_signal(scheme):
        mu = np.zeros(n)
        for i, tkr in enumerate(all_tkrs):
            sign = 1.0 if tkr in long_set else -1.0
            if scheme == 'A_equal':
                mu[i] = sign
            elif scheme == 'B_mktcap':
                mc = mcap_snap.get(tkr, np.nan)
                mu[i] = sign * (mc if np.isfinite(mc) and mc > 0 else 0.0)
            elif scheme == 'C_confidence':
                mu[i] = sign * yr_conf.get(tkr, 0.0)
        # Normalise each leg independently so longs and shorts have equal gross signal
        long_mask  = np.array([t in long_set  for t in all_tkrs])
        short_mask = np.array([t in short_set for t in all_tkrs])
        for mask in [long_mask, short_mask]:
            leg_sum = np.abs(mu[mask]).sum()
            if leg_sum > 1e-10:
                mu[mask] /= leg_sum
        return mu

    # ── Optimise weights ──────────────────────────────────────────────────────
    weights  = {}
    opt_ok   = {}
    for scheme in SCHEMES:
        mu = make_signal(scheme)
        if np.abs(mu).sum() < 1e-10:
            weights[scheme] = np.zeros(n)
            opt_ok[scheme]  = False
        else:
            w, ok = build_weights(mu, B, omega_eps, Omega_F, sectors)
            weights[scheme] = w
            opt_ok[scheme]  = ok

    # ── Simulate P&L over holding period ─────────────────────────────────────
    hold_ret = returns.loc[(returns.index >= entry_date) &
                           (returns.index <= exit_date), all_tkrs]

    # Fill NaN returns with 0 (no P&L on missing days for a stock)
    hold_ret_filled = hold_ret.fillna(0.0)

    pnl_records = []
    for date, daily_rets in hold_ret_filled.iterrows():
        rec = {'date': date, 'trade_day': (hold_ret.index.get_loc(date) + ENTRY_OFFSET)}
        for scheme in SCHEMES:
            rec[f'pnl_{scheme}'] = float(weights[scheme] @ daily_rets.values)
        pnl_records.append(rec)

    pnl_df = pd.DataFrame(pnl_records).set_index('date')

    # ── Factor exposure drift (scheme A, fixed weights) ────────────────────────
    # Daily realised factor exposure = wᵀ (r_t - RF_t) projected on B
    # For drift plot: report Bᵀw at entry (static since weights don't rebalance)
    w_A         = weights['A_equal']
    factor_exp  = B.T @ w_A   # (5,) — static factor exposure of scheme A

    # Compute cumulative factor PnL contribution day-by-day for drift analysis
    ff5_hold    = FF5.loc[(FF5.index >= entry_date) & (FF5.index <= exit_date), FACTOR_COLS]
    factor_pnl  = pd.DataFrame(
        {f: factor_exp[k] * ff5_hold[f].values for k, f in enumerate(FACTOR_COLS)},
        index=ff5_hold.index
    )
    cum_factor_pnl = factor_pnl.cumsum()

    # ── Metadata ──────────────────────────────────────────────────────────────
    meta = {
        'year':          year,
        'rank_date':     rank_date.date(),
        'entry_date':    entry_date.date(),
        'exit_date':     exit_date.date(),
        'n_long':        len(long_set),
        'n_short':       len(short_set),
        'n_portfolio':   n,
        'mean_r2':       float(r2.mean()),
        'pca_top1_evr':  float(pca_evr[0]) if len(pca_evr) > 0 else np.nan,
        'factor_exp_A':  factor_exp,
        'opt_ok':        opt_ok,
    }

    return pnl_df, meta, cum_factor_pnl


print('simulate_year() defined.')

In [ ]:
# ── Main backtest loop ─────────────────────────────────────────────────────────
# NOTE: Signal mode = HINDSIGHT. Known add/delete events are used as signals.
# This shows theoretical max alpha — forward-looking bias acknowledged.

all_pnl   = {}   # year → pnl_df
all_meta  = {}   # year → meta dict
all_drift = {}   # year → cum_factor_pnl DataFrame

for year, row in calendar.iterrows():
    pnl_df, meta, drift = simulate_year(year, row)
    all_meta[year] = meta
    if 'skip' in meta:
        print(f'[{year}] SKIPPED — {meta["skip"]}')
        continue
    all_pnl[year]   = pnl_df
    all_drift[year] = drift
    # Per-scheme Sharpe over holding window
    sr_str = '  '.join(
        f'{s.split("_")[1][0].upper()}: SR={pnl_df[f"pnl_{s}"].mean() / (pnl_df[f"pnl_{s}"].std() + 1e-12) * np.sqrt(252):.2f}'
        for s in SCHEMES
    )
    opt_str = '  '.join(f'{s.split("_")[1][0].upper()}: {"✓" if meta["opt_ok"][s] else "✗"}' for s in SCHEMES)
    print(f'[{year}]  n_L={meta["n_long"]:3d}  n_S={meta["n_short"]:3d}  '
          f'mean_R²={meta["mean_r2"]:.3f}  opt=[{opt_str}]  {sr_str}')

print(f'\nCompleted {len(all_pnl)} / {len(calendar)} years.')

## Results

In [ ]:
# ── Annual Sharpe table ────────────────────────────────────────────────────────
rows = []
for year in sorted(all_pnl.keys()):
    pnl  = all_pnl[year]
    meta = all_meta[year]
    row  = {
        'Year'        : year,
        'n_long'      : meta['n_long'],
        'n_short'     : meta['n_short'],
        'mean_R²'     : round(meta['mean_r2'], 3),
    }
    for scheme in SCHEMES:
        col = f'pnl_{scheme}'
        ser = pnl[col]
        mu_d  = ser.mean()
        sd_d  = ser.std()
        ann_ret  = mu_d * 252
        ann_vol  = sd_d * np.sqrt(252)
        sr       = mu_d / (sd_d + 1e-12) * np.sqrt(252)
        cum      = (1 + ser).prod() - 1
        lbl = scheme.split('_')[1][0].upper()  # A, B, C
        row[f'SR_{lbl}']     = round(sr, 2)
        row[f'Ret_{lbl}']    = f'{ann_ret*100:.1f}%'
        row[f'Vol_{lbl}']    = f'{ann_vol*100:.1f}%'
        row[f'Cumret_{lbl}'] = f'{cum*100:.1f}%'
    rows.append(row)

summary = pd.DataFrame(rows).set_index('Year')

# Print Sharpe sub-table for readability
sr_cols = [c for c in summary.columns if c.startswith('SR_')]
print('=== Annual Sharpe Ratios (annualised, over holding window) ===')
print(summary[['n_long', 'n_short', 'mean_R²'] + sr_cols].to_string())
print()
print('=== Pooled ===')
for scheme in SCHEMES:
    pooled = pd.concat([all_pnl[y][f'pnl_{scheme}'] for y in all_pnl])
    sr = pooled.mean() / (pooled.std() + 1e-12) * np.sqrt(252)
    cum = (1 + pooled).prod() - 1
    print(f'  {SCHEME_LABELS[scheme]:28s}  SR={sr:.2f}  Cum={cum*100:.1f}%')

In [ ]:
# ── Cumulative P&L + Sharpe comparison ────────────────────────────────────────
fig, axes = plt.subplots(3, 1, figsize=(16, 12), sharex=False)

# ── Panel 1: Annual Sharpe by scheme ──────────────────────────────────────────
ax = axes[0]
years = sorted(all_pnl.keys())
x = np.arange(len(years))
w_bar = 0.25
for j, scheme in enumerate(SCHEMES):
    sr_vals = []
    for year in years:
        ser = all_pnl[year][f'pnl_{scheme}']
        sr_vals.append(ser.mean() / (ser.std() + 1e-12) * np.sqrt(252))
    ax.bar(x + (j - 1) * w_bar, sr_vals, width=w_bar,
           color=SCHEME_COLORS[scheme], label=SCHEME_LABELS[scheme], alpha=0.85)
ax.axhline(0, color='black', lw=0.7)
ax.set_xticks(x)
ax.set_xticklabels(years, rotation=45, ha='right')
ax.set_ylabel('Annualised Sharpe')
ax.set_title('Annual Sharpe by Weighting Scheme (holding window)')
ax.legend(loc='upper left')
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.1f'))

# ── Panel 2: Pooled cumulative P&L (per $1 long + $1 short) ───────────────────
ax = axes[1]
for scheme in SCHEMES:
    pooled = pd.concat([all_pnl[y][f'pnl_{scheme}'] for y in years]).sort_index()
    cum    = (1 + pooled).cumprod() - 1
    ax.plot(cum.values, color=SCHEME_COLORS[scheme], label=SCHEME_LABELS[scheme], lw=1.5)
ax.set_xlabel('Holding days (all years stacked)')
ax.set_ylabel('Cumulative return (on GROSS=2)')
ax.set_title('Pooled Cumulative P&L — all years stacked')
ax.legend()
ax.axhline(0, color='black', lw=0.7, ls='--')
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=1))

# ── Panel 3: Drawdown ─────────────────────────────────────────────────────────
ax = axes[2]
for scheme in SCHEMES:
    pooled  = pd.concat([all_pnl[y][f'pnl_{scheme}'] for y in years]).sort_index()
    cum_ret = (1 + pooled).cumprod()
    rolling_max = cum_ret.expanding().max()
    dd = (cum_ret - rolling_max) / rolling_max
    ax.fill_between(np.arange(len(dd)), dd.values, 0,
                    alpha=0.35, color=SCHEME_COLORS[scheme], label=SCHEME_LABELS[scheme])
ax.set_xlabel('Holding days (all years stacked)')
ax.set_ylabel('Drawdown')
ax.set_title('Pooled Drawdown')
ax.legend()
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=1))

fig.tight_layout()
plt.show()

In [ ]:
# ── Factor exposure at entry (scheme A) — cross-year heatmap ──────────────────
exp_rows = []
for year in sorted(all_meta.keys()):
    m = all_meta[year]
    if 'factor_exp_A' not in m:
        continue
    row = dict(zip(FACTOR_COLS, m['factor_exp_A']))
    row['Year'] = year
    exp_rows.append(row)

exp_df = pd.DataFrame(exp_rows).set_index('Year')[FACTOR_COLS]

fig, ax = plt.subplots(figsize=(10, 7))
vmax = max(0.01, float(np.abs(exp_df.values).max()))
sns.heatmap(
    exp_df, ax=ax, center=0, cmap='RdBu_r',
    vmin=-vmax, vmax=vmax,
    linewidths=0.5, annot=True, fmt='.4f', annot_kws={'size': 8}
)
ax.set_title('FF5 Factor Exposure at Entry — Scheme A (Equal Weight)\n'
             '(Bᵀw: residual exposure after FF5-neutral constraint)', fontsize=11)
ax.set_xlabel('Factor')
ax.set_ylabel('Year')
plt.tight_layout()
plt.show()

print('Note: values near zero confirm factor-neutral constraint was binding.')
print('Non-zero values indicate failed optimisation for that year (fallback used).')

In [ ]:
# ── Factor exposure drift over holding period ─────────────────────────────────
# Plot cumulative factor P&L contribution by factor, averaged across years.
# A flat line = factor remained neutral throughout holding period.
# Drift = systematic exposure grew as returns accumulated (weights not rebalanced).

# Align all years by trade day (0 = entry, 80 = exit)
n_hold = EXIT_OFFSET - ENTRY_OFFSET   # total holding days
aligned = {f: [] for f in FACTOR_COLS}

for year, drift_df in all_drift.items():
    drift_arr = drift_df.values   # shape (hold_days, 5)
    # Pad or trim to n_hold
    padded = np.full((n_hold, 5), np.nan)
    T = min(len(drift_arr), n_hold)
    padded[:T] = drift_arr[:T]
    for k, f in enumerate(FACTOR_COLS):
        aligned[f].append(padded[:, k])

fig, axes = plt.subplots(1, 5, figsize=(20, 4), sharey=True)
t_axis = np.arange(ENTRY_OFFSET, ENTRY_OFFSET + n_hold)

for ax, f in zip(axes, FACTOR_COLS):
    mat  = np.array(aligned[f])       # (n_years, n_hold)
    mean = np.nanmean(mat, axis=0)
    std  = np.nanstd(mat, axis=0)
    ax.plot(t_axis, mean, lw=2, color='#1565C0', label='Mean across years')
    ax.fill_between(t_axis, mean - std, mean + std,
                    alpha=0.2, color='#1565C0', label='±1 SD')
    ax.axvline(0, color='black',  ls='--', lw=1,   label='Rank date (t=0)')
    ax.axhline(0, color='gray',   ls=':',  lw=0.8)
    ax.set_title(f)
    ax.set_xlabel('Trade day relative to rank date')
    if ax == axes[0]:
        ax.set_ylabel('Cumulative factor P&L contribution ($)')
        ax.legend(fontsize=8)

fig.suptitle('Factor P&L Drift Over Holding Period — Scheme A (Equal Weight)',
             fontsize=12, y=1.02)
fig.tight_layout()
plt.show()

In [ ]:
# ── P&L decomposition: pre-rank vs post-rank ───────────────────────────────────
# Split each year's holding period at t=0 (rank date) and compare pre vs post P&L.
# If the strategy works, pre-rank P&L >> post-rank P&L (drift happens before reconstitution).

decomp_rows = []
for year in sorted(all_pnl.keys()):
    pnl  = all_pnl[year]
    meta = all_meta[year]
    rank_date = pd.Timestamp(meta['rank_date'])
    for scheme in SCHEMES:
        col    = f'pnl_{scheme}'
        pre_pnl  = pnl.loc[pnl.index < rank_date, col].sum()
        post_pnl = pnl.loc[pnl.index >= rank_date, col].sum()
        decomp_rows.append({
            'Year': year, 'Scheme': SCHEME_LABELS[scheme],
            'Pre-rank P&L': round(pre_pnl, 5),
            'Post-rank P&L': round(post_pnl, 5),
            'Total P&L': round(pre_pnl + post_pnl, 5),
        })

decomp = pd.DataFrame(decomp_rows)

# Plot: pre vs post P&L by year, for scheme A
fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=False)
for ax, scheme in zip(axes, SCHEMES):
    sub = decomp[decomp['Scheme'] == SCHEME_LABELS[scheme]]
    x   = np.arange(len(sub))
    ax.bar(x - 0.2, sub['Pre-rank P&L'].values,  width=0.4,
           color='#2E7D32', alpha=0.85, label='Pre-rank (t=−60→0)')
    ax.bar(x + 0.2, sub['Post-rank P&L'].values, width=0.4,
           color='#E65100', alpha=0.85, label='Post-rank (t=0→+20)')
    ax.set_xticks(x)
    ax.set_xticklabels(sub['Year'].values, rotation=45, ha='right')
    ax.axhline(0, color='black', lw=0.7)
    ax.set_title(SCHEME_LABELS[scheme])
    ax.set_ylabel('P&L (per $2 gross)') if ax == axes[0] else None
    ax.legend(fontsize=8)

fig.suptitle('Pre-rank vs Post-rank P&L Decomposition by Year', fontsize=13)
fig.tight_layout()
plt.show()

# Summary table
print('\n=== Mean Pre/Post P&L across years, by scheme ===')
print(decomp.groupby('Scheme')[['Pre-rank P&L','Post-rank P&L','Total P&L']].mean().to_string())

In [ ]:
# ── PCA residual diagnostics — top eigenvalue across years ────────────────────
# Top-1 explained variance ratio of the residual covariance matrix.
# Low (<15%) = residuals are well-diversified after FF5 removal (good model fit).
# High (>25%) = residual factor structure remains — FF5 is insufficient.

pca_years = sorted(y for y, m in all_meta.items() if 'pca_top1_evr' in m and not np.isnan(m['pca_top1_evr']))
pca_vals  = [all_meta[y]['pca_top1_evr'] for y in pca_years]

fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(pca_years, [v * 100 for v in pca_vals], color='#5C6BC0', alpha=0.85)
ax.axhline(15, color='green', ls='--', lw=1.2, label='15% threshold (well-diversified)')
ax.axhline(25, color='red',   ls='--', lw=1.2, label='25% threshold (residual factor present)')
ax.set_xlabel('Year')
ax.set_ylabel('Top PC explained variance (%) — OLS residuals')
ax.set_title('PCA Residual Diagnostics: How much variance remains after FF5 removal?')
ax.legend()
fig.tight_layout()
plt.show()

In [ ]:
# ── Scheme comparison summary ──────────────────────────────────────────────────
print('=' * 70)
print('POOLED PERFORMANCE SUMMARY  (all years combined)')
print('=' * 70)
print(f'{"":<30} {"Sharpe":>8} {"Ann Ret":>8} {"Ann Vol":>8} {"Max DD":>8} {"Win Rate":>9}')
print('-' * 70)

for scheme in SCHEMES:
    pooled = pd.concat([all_pnl[y][f'pnl_{scheme}'] for y in sorted(all_pnl.keys())]).sort_index()
    mu_d   = pooled.mean()
    sd_d   = pooled.std()
    sr     = mu_d / (sd_d + 1e-12) * np.sqrt(252)
    ann_ret= mu_d * 252
    ann_vol= sd_d * np.sqrt(252)
    cum    = (1 + pooled).cumprod()
    dd     = (cum - cum.expanding().max()) / cum.expanding().max()
    max_dd = dd.min()
    win_rate = (pooled > 0).mean()
    print(f'{SCHEME_LABELS[scheme]:<30} {sr:>8.2f} {ann_ret*100:>7.1f}% {ann_vol*100:>7.1f}% '
          f'{max_dd*100:>7.1f}% {win_rate*100:>8.1f}%')

print('=' * 70)
print()
print('⚠  LOOKAHEAD BIAS WARNING: Signal uses known reconstitution events.')
print('   For realistic alpha, replace with prediction-mode signal from')
print('   the factor-ARIMA rank predictions (factor_arima_cache/).')